# 02 — Cleaning & Wrangling

**Stage 2 of the Tier A pipeline.** Every change is a line in a cleaning log with the rows affected
and the reason. Nothing is dropped for being inconvenient; the long tail (one-third of rows below
100k units) stays, because the concentration question in `03`–`04` is *about* the tail.

| § | Task | Rule |
|---|---|---|
| 2.1 | Repeated keys | four keys, four rules — one dropped duplicate, one dropped stray, one **merge**, one kept pair |
| 2.2 | Impossible years | pre-launch and post-snapshot years → missing |
| 2.3 | Missing year | recover from the same title on another platform **only** when its sibling years agree within 1 year; flag every imputation |
| 2.4 | Publisher | NaN and `"Unknown"` unified; recover from siblings when unambiguous; flag |
| 2.5 | Types | `Year` → nullable int; categoricals |
| 2.6 | Derived columns | manufacturer, platform type, generation, era, first-party flag, hit flags, region shares, title-level aggregates, censoring flag |
| 2.7 | Business-rule validation | assertions after cleaning, at a ±0.02 reconciliation tolerance |

In [1]:
import json, warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

ROOT = Path.cwd()
while not (ROOT / "archive" / "vgsales.csv").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
ARCHIVE = ROOT / "archive"
RAW, PROC = ROOT / "data" / "raw", ROOT / "data" / "processed"
REPORTS, FIGS, MODELS = ROOT / "reports", ROOT / "reports" / "figures", ROOT / "models"
for p in (RAW, PROC, REPORTS, FIGS, MODELS):
    p.mkdir(parents=True, exist_ok=True)

KF_PATH = REPORTS / "_key_figures.json"


def _plain(v):
    if hasattr(v, "item"):
        return v.item()
    if isinstance(v, (list, tuple)):
        return [_plain(x) for x in v]
    return v


def kf_update(**kw):
    """Append to reports/_key_figures.json — the only place the report reads numbers from."""
    kf = json.loads(KF_PATH.read_text()) if KF_PATH.exists() else {}
    kf.update({k: _plain(v) for k, v in kw.items()})
    KF_PATH.write_text(json.dumps(kf, indent=1, sort_keys=True))
    print(f"  _key_figures.json <- {', '.join(kw)}  ({len(kf)} figures total)")

df = pd.read_csv(RAW / "vgsales_raw_immutable.csv", encoding="utf-8")
REGIONS = ["NA_Sales", "EU_Sales", "JP_Sales", "Other_Sales"]
LOG = []


def log(step, rows, action, reason):
    LOG.append({"step": step, "rows_affected": int(rows), "action": action, "reason": reason})
    print(f"[{step}] {rows:>4} rows · {action}")


n0 = len(df)
print(f"loaded {n0:,} rows")

loaded 16,598 rows


## 2.1 Repeated keys — four keys, four rules

Diagnosed in `01` §1.5. The rules are keyed on `Rank` so they are auditable and cannot silently
match anything else.

In [2]:
# (a) Wii de Asobu: Metroid Prime — exact duplicate apart from Rank
assert df.loc[df.Rank.isin([15000, 15002])].drop(columns="Rank").duplicated(keep=False).all()
df = df[df.Rank != 15002]
log("2.1a", 1, "drop rank 15002 (Wii de Asobu: Metroid Prime, Wii)", "exact duplicate of rank 15000 excluding Rank")

# (b) Madden NFL 13 PS3 — a 0.01 M stray beside the 2.56 M record, same year and publisher
assert df.loc[df.Rank == 16130, "Global_Sales"].item() == 0.01
df = df[df.Rank != 16130]
log("2.1b", 1, "drop rank 16130 (Madden NFL 13, PS3, 0.01 M)", "scraper stray of rank 604; same title/platform/year/publisher")

# (c) Sonic the Hedgehog PS3 — the stray holds 0.48 M of EU units the main row lacks: MERGE
main, stray = df.index[df.Rank == 1717][0], df.index[df.Rank == 4147][0]
for c in REGIONS + ["Global_Sales"]:
    df.loc[main, c] = round(df.loc[main, c] + df.loc[stray, c], 2)
df = df.drop(index=stray)
log("2.1c", 1, "merge rank 4147 into rank 1717 (Sonic the Hedgehog, PS3)",
    "split record — stray is EU-only with no year/publisher; dropping it would lose 0.48 M real units")
print(df.loc[[main], ["Name", "Platform", "Year", *REGIONS, "Global_Sales"]].to_string())

# (d) Need for Speed: Most Wanted — 2005 original and 2012 reboot on X360 and PC: legitimately distinct
nfs = df[(df.Name == "Need for Speed: Most Wanted") & df.Platform.isin(["X360", "PC"])]
assert sorted(nfs.Year) == [2005.0, 2005.0, 2012.0, 2012.0]
log("2.1d", 0, "keep both Need for Speed: Most Wanted pairs", "two different games (2005, 2012) sharing a name")

[2.1a]    1 rows · drop rank 15002 (Wii de Asobu: Metroid Prime, Wii)
[2.1b]    1 rows · drop rank 16130 (Madden NFL 13, PS3, 0.01 M)
[2.1c]    1 rows · merge rank 4147 into rank 1717 (Sonic the Hedgehog, PS3)
                    Name Platform    Year  NA_Sales  EU_Sales  JP_Sales  Other_Sales  Global_Sales
1715  Sonic the Hedgehog      PS3  2006.0      0.41      0.55      0.04         0.66          1.66
[2.1d]    0 rows · keep both Need for Speed: Most Wanted pairs


## 2.2 Impossible years → missing

A release year before the platform existed, or after the 2016 snapshot, is a data error, not a
release. Platform launch years are external reference facts (first launch in any region).

In [3]:
PLATFORM_LAUNCH = {
    "2600": 1977, "NES": 1983, "SNES": 1990, "N64": 1996, "GC": 2001, "Wii": 2006, "WiiU": 2012,
    "GB": 1989, "GBA": 2001, "DS": 2004, "3DS": 2011, "PS": 1994, "PS2": 2000, "PS3": 2006,
    "PS4": 2013, "PSP": 2004, "PSV": 2011, "XB": 2001, "X360": 2005, "XOne": 2013, "GEN": 1988,
    "SAT": 1994, "DC": 1998, "GG": 1990, "SCD": 1991, "NG": 1990, "3DO": 1993, "TG16": 1987,
    "PCFX": 1994, "WS": 1999, "PC": np.nan,
}
launch = df.Platform.map(PLATFORM_LAUNCH)
bad_pre = df.Year.notna() & launch.notna() & (df.Year < launch)
bad_post = df.Year > 2016
print(df.loc[bad_pre | bad_post, ["Rank", "Name", "Platform", "Year"]].to_string(index=False))
df.loc[bad_pre | bad_post, "Year"] = np.nan
log("2.2a", bad_pre.sum(), "Year -> missing (released before platform launch)", "impossible: e.g. a DS game dated 1985")
log("2.2b", bad_post.sum(), "Year -> missing (dated after the 2016 snapshot)", "out of the file's window; includes a DS game dated 2020")
df["year_invalid_flag"] = (bad_pre | bad_post).astype(int)

 Rank                                             Name Platform   Year
 1352                               Disney's DuckTales       GB 1988.0
 2085                                   NFL Fever 2002       XB 2000.0
 5959                           Imagine: Makeup Artist       DS 2020.0
12272           ESPN Winter X-Games: Snowboarding 2002      GBA 2000.0
14393 Phantasy Star Online 2 Episode 4: Deluxe Package      PS4 2017.0
15870              Strongest Tokyo University Shogi DS       DS 1985.0
16244 Phantasy Star Online 2 Episode 4: Deluxe Package      PSV 2017.0
16441                 Brothers Conflict: Precious Baby      PSV 2017.0
[2.2a]    4 rows · Year -> missing (released before platform launch)
[2.2b]    4 rows · Year -> missing (dated after the 2016 snapshot)


## 2.3 Missing `Year` — recover from siblings, conservatively

The review notes missing years are not random (they cluster in 2004–2016 rows). A title released on
several platforms usually launched on them within a year of each other, so when **every known
sibling year agrees within 1 year** the earliest is a defensible estimate. Where siblings disagree
(ports years apart), the sibling year predates this platform's launch (a later port), or there is
no sibling, the year stays missing and the row is excluded only from
time-based analyses. Every imputation is flagged.

Years embedded in names (*Madden NFL 2004*) are **not** used: annual sports titles ship the autumn
*before* the year in their name, so that rule would be wrong by one year for exactly the rows it hits.

In [4]:
miss = df.Year.isna()
sib = df[df.Year.notna()].groupby("Name").Year.agg(["min", "max", "count"])
cand = df.loc[miss, "Name"].map(lambda n: sib.loc[n] if n in sib.index else None)

recov = pd.Series(np.nan, index=df.index)
n_sib_prelaunch = 0
for i, row in cand.items():
    if row is not None and (row["max"] - row["min"]) <= 1:
        if launch[i] > row["min"]:          # sibling year predates THIS platform: a later port
            n_sib_prelaunch += 1
            continue
        recov[i] = row["min"]
n_imp = int(recov.notna().sum())
n_sib_disagree = int(sum(1 for r in cand if r is not None) - n_imp - n_sib_prelaunch)
print(f"sibling year predates this platform's launch (port) — not imputed: {n_sib_prelaunch}")
df["year_imputed_flag"] = recov.notna().astype(int)
df.loc[recov.notna(), "Year"] = recov[recov.notna()]
print(f"missing Year before: {miss.sum()}  ·  recovered from siblings: {n_imp}  ·  "
      f"siblings disagree: {n_sib_disagree}  ·  no sibling: {miss.sum() - n_imp - n_sib_disagree - n_sib_prelaunch}")
log("2.3", n_imp, "Year imputed from same-title siblings (range ≤ 1 yr, earliest taken)",
    "multi-platform releases launch together; flagged year_imputed_flag")
log("2.3b", df.Year.isna().sum(), "Year left missing", "no sibling or siblings disagree; excluded from time analyses only")

sibling year predates this platform's launch (port) — not imputed: 2
missing Year before: 277  ·  recovered from siblings: 117  ·  siblings disagree: 5  ·  no sibling: 153
[2.3]  117 rows · Year imputed from same-title siblings (range ≤ 1 yr, earliest taken)
[2.3b]  160 rows · Year left missing


## 2.4 Publisher — unify NaN and "Unknown", recover when unambiguous

In [5]:
pub_missing = df.Publisher.isna() | (df.Publisher == "Unknown")
print(f"NaN: {df.Publisher.isna().sum()}  'Unknown': {(df.Publisher == 'Unknown').sum()}  -> {pub_missing.sum()} unusable")
known = df[~pub_missing].groupby("Name").Publisher.agg(lambda s: s.unique().tolist())
fill = df.loc[pub_missing, "Name"].map(lambda n: known[n][0] if n in known.index and len(known[n]) == 1 else None)
fill = fill.dropna()
df["publisher_imputed_flag"] = 0
df.loc[fill.index, "Publisher"] = fill
df.loc[fill.index, "publisher_imputed_flag"] = 1
df["Publisher"] = df.Publisher.fillna("Unknown")
log("2.4a", len(fill), "Publisher imputed from same-title siblings (single known publisher)", "flagged publisher_imputed_flag")
log("2.4b", (df.Publisher == "Unknown").sum(), "Publisher NaN + 'Unknown' unified as 'Unknown'",
    "one explicit category, never silently dropped")

NaN: 57  'Unknown': 203  -> 260 unusable


[2.4a]   57 rows · Publisher imputed from same-title siblings (single known publisher)
[2.4b]  203 rows · Publisher NaN + 'Unknown' unified as 'Unknown'


## 2.5 Types

In [6]:
df["Year"] = df.Year.astype("Int64")
for c in ["Platform", "Genre"]:
    df[c] = df[c].astype("category")
log("2.5", len(df), "Year -> Int64; Platform, Genre -> category", "enforce types early")
print(df.dtypes.to_string())

[2.5] 16595 rows · Year -> Int64; Platform, Genre -> category
Rank                         int64
Name                           str
Platform                  category
Year                         Int64
Genre                     category
Publisher                      str
NA_Sales                   float64
EU_Sales                   float64
JP_Sales                   float64
Other_Sales                float64
Global_Sales               float64
year_invalid_flag            int64
year_imputed_flag            int64
publisher_imputed_flag       int64


## 2.6 Derived columns

All are deterministic functions of the row (or of a title's rows) — **no target information is
encoded into any feature here**; the publisher track-record features that *do* use outcomes are
built in `05`, strictly from earlier years.

Eras follow console-generation turnovers so that each era is one competitive landscape:
**1980–94** (8/16-bit), **1995–2000** (PlayStation / N64), **2001–05** (PS2 / Xbox / GameCube),
**2006–10** (Wii / X360 / PS3 / DS ascendancy), **2011–16** (3DS / PS4 / XOne; mobile substitution).

In [7]:
MANUFACTURER = {
    **{p: "Nintendo" for p in ["NES", "SNES", "N64", "GC", "Wii", "WiiU", "GB", "GBA", "DS", "3DS"]},
    **{p: "Sony" for p in ["PS", "PS2", "PS3", "PS4", "PSP", "PSV"]},
    **{p: "Microsoft" for p in ["XB", "X360", "XOne"]},
    **{p: "Sega" for p in ["GEN", "SAT", "DC", "GG", "SCD"]},
    "PC": "PC", "2600": "Other", "NG": "Other", "3DO": "Other", "TG16": "Other", "PCFX": "Other", "WS": "Other",
}
HANDHELD = {"GB", "GBA", "DS", "3DS", "PSP", "PSV", "GG", "WS"}
GENERATION = {
    "2600": 2, "NES": 3, "SNES": 4, "GEN": 4, "TG16": 4, "SCD": 4, "NG": 4, "GB": 4, "GG": 4,
    "PS": 5, "SAT": 5, "N64": 5, "3DO": 5, "PCFX": 5, "WS": 5,
    "PS2": 6, "XB": 6, "GC": 6, "DC": 6, "GBA": 6,
    "PS3": 7, "X360": 7, "Wii": 7, "DS": 7, "PSP": 7,
    "PS4": 8, "XOne": 8, "WiiU": 8, "3DS": 8, "PSV": 8, "PC": np.nan,
}
FIRST_PARTY_PUBLISHERS = {
    "Nintendo": "Nintendo",
    "Sony Computer Entertainment": "Sony", "Sony Computer Entertainment Europe": "Sony",
    "Sony Computer Entertainment America": "Sony",
    "Microsoft Game Studios": "Microsoft", "Sega": "Sega",
}
plat = df.Platform.astype(str)
df["manufacturer"] = plat.map(MANUFACTURER)
df["platform_type"] = np.where(plat == "PC", "PC", np.where(plat.isin(HANDHELD), "Handheld", "Home console"))
df["generation"] = plat.map(GENERATION).astype("Int64")
df["platform_launch_year"] = plat.map(PLATFORM_LAUNCH).astype("Int64")
df["years_since_launch"] = (df.Year - df.platform_launch_year).astype("Int64")
df["first_party"] = (df.Publisher.map(FIRST_PARTY_PUBLISHERS) == df.manufacturer).astype(int)

ERA_BINS, ERA_LABELS = [1979, 1994, 2000, 2005, 2010, 2016], ["1980–94", "1995–2000", "2001–05", "2006–10", "2011–16"]
df["era"] = pd.cut(df.Year.astype(float), bins=ERA_BINS, labels=ERA_LABELS)

g = df.Global_Sales
df["is_hit"] = (g >= 1).astype(int)
df["is_blockbuster"] = (g >= 10).astype(int)
df["log_sales"] = np.log1p(g)
for c, short in zip(REGIONS, ["na", "eu", "jp", "other"]):
    df[f"{short}_share"] = (df[c] / df[REGIONS].sum(axis=1)).fillna(0)
df["dominant_region"] = df[REGIONS].idxmax(axis=1).str.replace("_Sales", "")
df["n_platforms_title"] = df.groupby("Name").Platform.transform("nunique")
df["is_multiplatform"] = (df.n_platforms_title > 1).astype(int)
# Lifetime sales are right-censored for recent releases: by the late-2016 snapshot a 2015 game has
# had ~1–2 years on sale and a 2016 game < 1. Flag them; every era/trend view treats them separately.
df["censored_flag"] = (df.Year >= 2015).fillna(False).astype(int)
df["in_time_window"] = df.Year.notna().astype(int)

assert df.manufacturer.notna().all()
print(df.groupby("manufacturer").agg(rows=("Name", "size"), units=("Global_Sales", "sum"),
                                     first_party_rows=("first_party", "sum")).round(0).to_string())
print(f"\nfirst-party rows: {df.first_party.sum():,} ({df.first_party.mean():.1%}) · "
      f"units {df.loc[df.first_party == 1, 'Global_Sales'].sum():,.0f} M "
      f"({df.loc[df.first_party == 1, 'Global_Sales'].sum() / g.sum():.1%})")
log("2.6", len(df), "derived: manufacturer, platform_type, generation, era, first_party, hit flags, "
    "region shares, title aggregates, censored_flag", "deterministic; no target leakage")

              rows   units  first_party_rows
manufacturer                                
Microsoft     2302  1379.0               159
Nintendo      6271  3522.0               704
Other          157   100.0                 0
PC             960   259.0                 0
Sega           259    80.0               132
Sony          6646  3580.0               701

first-party rows: 1,696 (10.2%) · units 2,710 M (30.4%)
[2.6] 16595 rows · derived: manufacturer, platform_type, generation, era, first_party, hit flags, region shares, title aggregates, censored_flag


### Title-level table

The same game on PS3 and X360 is two rows. Concentration and hit-rate claims are checked at **both**
units (`03` §B3); the title table is the second unit.

In [8]:
titles = (df.groupby("Name", observed=True)
            .agg(units=("Global_Sales", "sum"), first_year=("Year", "min"),
                 n_platforms=("Platform", "nunique"),
                 genre=("Genre", lambda s: s.mode().iat[0]),
                 publisher=("Publisher", lambda s: s.mode().iat[0]),
                 first_party_any=("first_party", "max"),
                 **{c: (c, "sum") for c in REGIONS})
            .reset_index())
titles["is_hit"] = (titles.units >= 1).astype(int)
print(f"{len(titles):,} titles · {titles.is_hit.sum():,} title-level hits ({titles.is_hit.mean():.1%}) "
      f"vs {df.is_hit.sum():,} row-level hits ({df.is_hit.mean():.1%})")
titles.sort_values("units", ascending=False).head(8)

11,493 titles · 2,041 title-level hits (17.8%) vs 2,081 row-level hits (12.5%)


,Name,units,first_year,n_platforms,genre,publisher,first_party_any,NA_Sales,EU_Sales,JP_Sales,Other_Sales,is_hit
11007,Wii Sports,82.74,2006,1,Sports,Nintendo,1,41.49,29.02,3.77,8.46,1
3712,Grand Theft Auto V,55.92,2013,5,Action,Take-Two Interactive,0,23.46,23.04,1.39,8.03,1
9327,Super Mario Bros.,45.31,1985,2,Platform,Nintendo,1,32.48,4.88,6.96,0.99,1
9715,Tetris,35.84,1988,2,Puzzle,Nintendo,1,26.17,2.95,6.03,0.69,1
5573,Mario Kart Wii,35.82,2008,1,Racing,Nintendo,1,15.85,12.88,3.79,3.31,1
11009,Wii Sports Resort,33.00,2009,1,Sports,Nintendo,1,15.75,11.01,3.28,2.96,1
7346,Pokemon Red/Pokemon Blue,31.37,1996,1,Role-Playing,Nintendo,1,11.27,8.89,10.22,1.00,1
1222,Call of Duty: Black Ops,31.03,2010,5,Shooter,Activision,0,17.59,9.50,0.59,3.36,1


## 2.7 Business-rule validation (post-cleaning)

In [9]:
resid = (df[REGIONS].sum(axis=1) - df.Global_Sales).round(2)
rules = {
    "no repeated (Name, Platform, Year) keys": not df.duplicated(["Name", "Platform", "Year"]).any(),
    "Global == ΣRegions within ±0.02": bool((resid.abs() <= 0.02).all()),
    "no year before platform launch": bool(((df.Year >= df.platform_launch_year) | df.Year.isna()
                                            | df.platform_launch_year.isna()).all()),
    "no year after 2016": bool((df.Year.dropna() <= 2016).all()),
    "Publisher has no NaN": bool(df.Publisher.notna().all()),
    "every row has a manufacturer": bool(df.manufacturer.notna().all()),
    "region shares sum to 1 where sales > 0": bool(np.allclose(
        df.loc[df[REGIONS].sum(axis=1) > 0, ["na_share", "eu_share", "jp_share", "other_share"]].sum(axis=1), 1)),
    "units conserved (only 2 dropped rows' units lost)": abs(df.Global_Sales.sum()
        - pd.read_csv(RAW / "vgsales_raw_immutable.csv").Global_Sales.sum() + 0.02 + 0.01) < 0.05,
}
for k, v in rules.items():
    print(f"  {'PASS' if v else 'FAIL'}  {k}")
assert all(rules.values())

  PASS  no repeated (Name, Platform, Year) keys
  PASS  Global == ΣRegions within ±0.02
  PASS  no year before platform launch
  PASS  no year after 2016
  PASS  Publisher has no NaN
  PASS  every row has a manufacturer
  PASS  region shares sum to 1 where sales > 0
  PASS  units conserved (only 2 dropped rows' units lost)


In [10]:
df.to_parquet(PROC / "vg_rows.parquet", index=False)
titles.to_parquet(PROC / "vg_titles.parquet", index=False)
log_df = pd.DataFrame(LOG)
log_df.to_csv(PROC / "_cleaning_log.csv", index=False)

kf_update(
    n_rows_clean=len(df), n_titles_clean=len(titles),
    n_rows_dropped=n0 - len(df), n_year_invalidated=int(df.year_invalid_flag.sum()),
    n_year_imputed=int(df.year_imputed_flag.sum()), n_year_missing_clean=int(df.Year.isna().sum()),
    n_pub_imputed=int(df.publisher_imputed_flag.sum()), n_pub_unknown_clean=int((df.Publisher == "Unknown").sum()),
    n_first_party_rows=int(df.first_party.sum()), first_party_row_share=float(df.first_party.mean()),
    first_party_unit_share=float(df.loc[df.first_party == 1, "Global_Sales"].sum() / df.Global_Sales.sum()),
    n_hits_clean=int(df.is_hit.sum()), hit_rate_rows=float(df.is_hit.mean()),
    n_title_hits=int(titles.is_hit.sum()), hit_rate_titles=float(titles.is_hit.mean()),
    n_censored_rows=int(df.censored_flag.sum()),
)
print()
log_df

  _key_figures.json <- n_rows_clean, n_titles_clean, n_rows_dropped, n_year_invalidated, n_year_imputed, n_year_missing_clean, n_pub_imputed, n_pub_unknown_clean, n_first_party_rows, first_party_row_share, first_party_unit_share, n_hits_clean, hit_rate_rows, n_title_hits, hit_rate_titles, n_censored_rows  (49 figures total)



,step,rows_affected,action,reason
0,2.1a,1,"drop rank 15002 (Wii de Asobu: Metroid Prime, ...",exact duplicate of rank 15000 excluding Rank
1,2.1b,1,"drop rank 16130 (Madden NFL 13, PS3, 0.01 M)",scraper stray of rank 604; same title/platform...
2,2.1c,1,merge rank 4147 into rank 1717 (Sonic the Hedg...,split record — stray is EU-only with no year/p...
3,2.1d,0,keep both Need for Speed: Most Wanted pairs,"two different games (2005, 2012) sharing a name"
4,2.2a,4,Year -> missing (released before platform launch),impossible: e.g. a DS game dated 1985
5,2.2b,4,Year -> missing (dated after the 2016 snapshot),out of the file's window; includes a DS game d...
6,2.3,117,Year imputed from same-title siblings (range ≤...,multi-platform releases launch together; flagg...
7,2.3b,160,Year left missing,no sibling or siblings disagree; excluded from...
8,2.4a,57,Publisher imputed from same-title siblings (si...,flagged publisher_imputed_flag
9,2.4b,203,Publisher NaN + 'Unknown' unified as 'Unknown',"one explicit category, never silently dropped"


## Stage 2 — Gate Checklist

- [x] Missing-value strategy documented **per column**: `Year` (sibling recovery with agreement rule,
      else missing + excluded from time views only), `Publisher` (sibling recovery, else explicit
      `"Unknown"`); both flagged
- [x] Duplicates assessed key by key — one dropped, one stray dropped, one **merged**, one kept, each justified
- [x] Dtypes enforced (`Year` nullable int; categoricals)
- [x] Business rules pass after cleaning, at the ±0.02 tolerance `01` established
- [x] Cleaning log written (`data/processed/_cleaning_log.csv`) — rows affected and reason per step
- [x] Cleaned data saved to `data/processed/` (row table + title table)